## C5_02 — Construirea vector store-ului pentru o bulă
În acest notebook construim un vector store FAISS pentru o singură bulă / un singur agent.
Fiecare student lucrează pe bula lui. Scopul este să vedem clar cum textele curățate devin embeddings, apoi index FAISS.
Mai târziu, aceeași logică va fi pusă într-un script `.py` care rulează automat pentru toate bulele.

## 0. Setup

^C


In [6]:
import sys
!{sys.executable} -m pip install sentence-transformers torch torchvision

  Using cached sentence_transformers-5.5.0-py3-none-any.whl.metadata (18 kB)
Using cached sentence_transformers-5.5.0-py3-none-any.whl (588 kB)



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
from pathlib import Path
import os, pickle
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer

while not Path("data/bubbles").exists():
    os.chdir("..")

BUBBLES_DIR = Path("data/bubbles")
VECTOR_DIR = Path("assets/vectorstores")
VECTOR_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"

## 1. Aleg bula mea
Alege fișierul `.jsonl` al bulei tale.
Acest fișier a fost creat în etapa anterioară, după verificarea manuală a textelor.

In [9]:
MY_BUBBLE_FILE = "personalist_salvator.jsonl" 

bubble_path = BUBBLES_DIR / MY_BUBBLE_FILE
slug = bubble_path.stem

df_bubble = pd.read_json(bubble_path, lines=True)

print("Bula:", slug)
print("Texte:", len(df_bubble))

df_bubble[["id", "agent", "text"]].head()

Bula: personalist_salvator
Texte: 2


,id,agent,text
0,yt_iH8jB4NlV9Y_UgwEkR3OE3CVbdLc33V4AaABAg,Personalist-salvator,Felicitări domnul Președinte George Simion Pre...
1,yt_Dv73fbPD2Vk_UgyAR6B64qSKb3jtnjN4AaABAg,Personalist-salvator,"POPORUL ROMÂN VĂ AȘTEAPTĂ, DOMNULE PREȘEDINTE,..."


## 2. Pregătim textele
Pentru FAISS avem nevoie de o listă simplă de texte.
Metadata rămâne separat, ca să putem lega fiecare vector de textul original.

In [10]:
texts = df_bubble["text"].fillna("").tolist()
metadata = df_bubble.to_dict(orient="records")

print("Primul text:")
print(texts[0][:500])

Primul text:
Felicitări domnul Președinte George Simion Președintele Românie AUR


## 3. Generăm embeddings
Un embedding este o reprezentare vectorială a textului: texte apropiate ca sens primesc vectori apropiați în spațiul semantic.
Folosim un model multilingv, deoarece corpusul este în limba română.
Normalizăm vectorii la lungime 1, astfel încât produsul scalar din FAISS să funcționeze ca similaritate cosinus.

In [11]:
model = SentenceTransformer(MODEL_NAME)
embeddings = model.encode(
    texts,
    normalize_embeddings=True,
    show_progress_bar=True
).astype("float32")
print("Număr texte:", len(texts))
print("Dimensiune embeddings:", embeddings.shape)

c:\Users\rotar\AppData\Local\Programs\Python\Python314\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\rotar\.cache\huggingface\hub\models--sentence-transformers--paraphrase-multilingual-MiniLM-L12-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Batches: 100%|██████████| 1/1 [00:00<00:00,  5.24it/s]

Număr texte: 2
Dimensiune embeddings: (2, 384)


### Verificare rapidă
Răspunde în 1–2 propoziții în notebook:
- Câte texte are bula ta?
- Câți vectori au fost generați?
- Ce înseamnă a doua valoare din `embeddings.shape`?

In [ ]:
# TODO student:
# Bula mea are 2 texte.
# Au fost generați 2 vectori.
# A doua valoare din embeddings.shape reprezintă dimensiunea fiecarui vector, adica fiecare embedding are 384 de valori numerice.

## 4. Construim indexul FAISS
FAISS este biblioteca care caută rapid vectori apropiați.
Indexul nu păstrează textele originale. El păstrează doar reprezentările vectoriale.
De aceea salvăm două lucruri:
- `index.faiss` = indexul vectorial;
- `index.pkl` = textele originale și metadatele.

In [12]:
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)
out_dir = VECTOR_DIR / slug
out_dir.mkdir(parents=True, exist_ok=True)
faiss.write_index(index, str(out_dir / "index.faiss"))
with open(out_dir / "index.pkl", "wb") as f:
    pickle.dump(metadata, f)
print("Salvat în:", out_dir)
print("Vectori în index:", index.ntotal)

Salvat în: assets\vectorstores\personalist_salvator
Vectori în index: 2


## 5. Verificăm fișierele create
Dacă totul a mers corect, bula ta are acum un folder propriu în `assets/vectorstores/`.
Acest folder trebuie să conțină `index.faiss` și `index.pkl`.

In [ ]:
# TODO student:
# index.faiss există: DA
# index.pkl există: DA
# index.ntotal este egal cu numărul de texte: 2

## Ce am construit?
Am transformat textele curate ale unei bule într-un index vectorial local.
Acest index nu generează răspunsuri. El doar permite căutarea semantică.
În următorul continuare vom testa dacă, pentru o întrebare, FAISS returnează texte relevante.

## 6. Testăm retrieval-ul
Acum simulăm logica aplicației.
- Utilizatorul introduce o știre sau o afirmație politică.
- Retriever-ul caută în memoria bulei cele mai asemănătoare texte.
- Nu generăm încă un răspuns cu LLM. Doar verificăm ce exemple sunt recuperate.

In [14]:
# Text nou introdus în aplicație

input_text = "Calin Georgescu ne va salva. El este presedintele ales."

In [15]:
# Transformăm textul nou în embedding

query_vector = model.encode(
    [input_text],
    normalize_embeddings=True
).astype("float32")

In [11]:
# query_vector

In [16]:
# Căutăm cele mai apropiate 5 texte din bula noastră

scores, results = index.search(query_vector, k=5)

for rank, pos in enumerate(results[0], start=1):
    row = metadata[pos]
    
    print(f"\nRezultat {rank}")
    print("Scor:", round(float(scores[0][rank-1]), 3))
    print("Text:", row["text"][:500])


Rezultat 1
Scor: 0.577
Text: Felicitări domnul Președinte George Simion Președintele Românie AUR

Rezultat 2
Scor: 0.278
Text: POPORUL ROMÂN VĂ AȘTEAPTĂ, DOMNULE PREȘEDINTE,,,,SĂ FACEM ROMÂNIA MARE,,,,

Rezultat 3
Scor: -3.4028234663852886e+38
Text: POPORUL ROMÂN VĂ AȘTEAPTĂ, DOMNULE PREȘEDINTE,,,,SĂ FACEM ROMÂNIA MARE,,,,

Rezultat 4
Scor: -3.4028234663852886e+38
Text: POPORUL ROMÂN VĂ AȘTEAPTĂ, DOMNULE PREȘEDINTE,,,,SĂ FACEM ROMÂNIA MARE,,,,

Rezultat 5
Scor: -3.4028234663852886e+38
Text: POPORUL ROMÂN VĂ AȘTEAPTĂ, DOMNULE PREȘEDINTE,,,,SĂ FACEM ROMÂNIA MARE,,,,


### TODO
Schimbă `input_text` cu o afirmație potrivită pentru agentul tău.
Rulează căutarea.
Notează:
- câte rezultate din 5 sunt relevante;
- dacă textele recuperate exprimă vocea agentului;
- dacă ai observat un text slab care ar trebui eliminat.

In [ ]:
# Am avut numai 2 rezultate relevante, restul s-au repetat din cauza ca nu au existat multe rezultate care sa rezoneze cu tipologia de agent.
# Textele reflecta vocea agentului.
# A fost eliminat un singur text care nu avea relevanta cu personalitatea agentului. 